# Train the temporal heatmap detector on Colab

Colab GPUs and session limits change over time, so the trainer checkpoints every epoch to
`out_dir/last.pt` on Google Drive and resumes automatically after a disconnect.

1. Upload the `Video_CV` repository (or `git clone` it from your GitHub) and the frame stores made with
   `rso make-dataset` to Google Drive, for example under `MyDrive/Video_CV/` and `MyDrive/Video_CV/data/datasets/`.
2. Runtime -> Change runtime type -> GPU.
3. Run the cells in order. Run the pilot first and note `sec_per_step` and `max_mem_gib`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Video_CV
!nvidia-smi

In [ ]:
!pip -q install -e ".[train]"
import torch; print(torch.__version__, torch.cuda.is_available())

In [ ]:
# Google Drive is slow for many small files: copy the frame stores to local disk first
!mkdir -p /content/datasets && rsync -a --info=progress2 data/datasets/ /content/datasets/
import yaml
cfg = yaml.safe_load(open('configs/train_heatmap.yaml'))
cfg['stores'] = ['/content/datasets/' + s.rstrip('/').split('/')[-1] for s in cfg['stores']]
yaml.safe_dump(cfg, open('configs/train_heatmap_colab.yaml', 'w'), sort_keys=False)
print(cfg['stores'])


In [ ]:
# 200-step pilot: speed and memory on this GPU
!python -m starship_rso.train.train_heatmap --config configs/train_heatmap_colab.yaml --pilot --set out_dir=runs/pilot
!cat runs/pilot/pilot.json

In [ ]:
# full training (resumes from runs/heatmap_k3/last.pt if the session was interrupted)
!python -m starship_rso.train.train_heatmap --config configs/train_heatmap_colab.yaml

In [ ]:
import pandas as pd
log = pd.read_csv('runs/heatmap_k3/log.csv'); log.tail(10)

Copy `runs/heatmap_k3/best.pt` to your Mac and run the live demo with it:

```
rso run -c configs/default.yaml -c configs/flight14.yaml --video data/videos/f14_deploy.mp4 \
    --set detector.type=fused --set detector.heatmap.weights=runs/heatmap_k3/best.pt
```

For the single-frame baseline and the 5-frame variant, repeat with `--set frames=1` / `--set frames=5`
and a different `out_dir`, then compare the `log.csv` files and `rso evaluate` results on the test split.